In [43]:
import sys
!{sys.executable} -m pip install -q yfinance pandas-datareader arch hmmlearn pykalman statsmodels xlsxwriter


In [44]:
import sys
!{sys.executable} -m pip install -q imblearn

In [45]:
from imblearn.over_sampling import SMOTE

In [61]:
import os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr
import re
import time

# Time-series models
from arch import arch_model
from hmmlearn import hmm
from pykalman import KalmanFilter
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, accuracy_score, f1_score)
from typing import List, Dict

OUTPUT_DIR = Path("/content/outputs_v27_timeseries_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
MIN_DATA_START = pd.Timestamp("2000-01-01")
TRAIN_START    = "2000-01-01"
TEST_DATE      = "2022-01-01"   # train < 2022, test 2022→2026

YF_CHUNK_SIZE        = 40
SLEEP_BETWEEN_CHUNKS = 1.0
MIN_COLUMN_COVERAGE  = 0.90
MAX_FIRST_VALID_LAG_DAYS = 365
ROLLING_QUANTILE_WINDOW  = 504

# Jours flat : conservés (aucun filtre)
FLAT_THRESHOLD_ABS = 0.0

HORIZONS = [1, 5]   # jours pour la prédiction directionnelle

np.random.seed(RANDOM_STATE)


In [47]:
# (select_and_filter_features est definie dans la cellule suivante - cette
# cellule etait un doublon residuel, neutralisee pour eviter la confusion.)


In [48]:
# =============================================================================
# Sélection de features par corrélation (reprise à l'identique du notebook
# original) : produit une LISTE ORDONNÉE de features (les plus corrélées au
# target en premier, en excluant les features trop corrélées entre elles).
# Cette liste classée sert ensuite de base pour tester N=1..20 features.
# =============================================================================

correlation_threshold_target = 0.05
correlation_threshold_features = 0.9

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    if not selected_features_initial:
        return []

    features_small = []
    high_corr_features = list(selected_features_initial)
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy()

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        correlated_with_f = []
        if f in df_for_inter_corr.columns:
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns:
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception:
                        pass

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    return features_small


In [49]:
# =============================================================================
# RECOMMANDATION 2 : génération systématique d'interactions explicites.
#
# Pour un ensemble de features de base (typiquement le top 30 actuel d'un
# régime/horizon), génère TOUTES les paires (ratio + différence) :
#   - ratio      : feat_i / feat_j  (nommé "{i}_div_{j}")
#   - différence : feat_i - feat_j  (nommé "{i}_minus_{j}")
#
# Pour 30 features : 30*29/2 = 435 paires non-ordonnées x 2 opérations = 870
# nouvelles colonnes potentielles. Toutes ne seront pas utiles - elles sont
# ensuite repassées dans select_and_filter_features (même filtre Spearman +
# anti-redondance) pour ne garder que celles qui apportent un signal réel.
#
# Protection : ratio i/j est mis à NaN si |j| < epsilon (évite divisions
# explosives qui pollueraient la sélection avec du bruit numérique).
# =============================================================================

def generate_pairwise_interactions(df: pd.DataFrame, base_features: list,
                                    epsilon: float = 1e-8) -> pd.DataFrame:
    """Génère ratio et différence pour toutes les paires non-ordonnées de
    base_features. Retourne un NOUVEAU dataframe (les colonnes générées
    uniquement, même index que df) - à concaténer par l'appelant."""
    n = len(base_features)
    print(f"[INTERACTIONS] Génération de paires pour {n} features de base "
          f"({n*(n-1)//2} paires x 2 opérations = {n*(n-1)} colonnes max)...")

    new_cols = {}
    for i in range(n):
        for j in range(i + 1, n):
            fi, fj = base_features[i], base_features[j]
            if fi not in df.columns or fj not in df.columns:
                continue

            col_i = df[fi]
            col_j = df[fj]

            # Différence
            diff_name = f"{fi}_minus_{fj}"
            new_cols[diff_name] = col_i - col_j

            # Ratio (protégé contre division par ~0)
            ratio_name = f"{fi}_div_{fj}"
            safe_denom = col_j.where(col_j.abs() >= epsilon, np.nan)
            new_cols[ratio_name] = col_i / safe_denom

    interactions_df = pd.DataFrame(new_cols, index=df.index)
    interactions_df = interactions_df.replace([np.inf, -np.inf], np.nan)
    print(f"[INTERACTIONS] {interactions_df.shape[1]} colonnes d'interactions générées.")
    return interactions_df


In [50]:
BAD_TICKERS = {
    "XXIV",
    "TVIX",
    "ZIV",
    "^MIB",
    "CELG",
    "AET",
    "HES",
    "GPS",
    "JWN",
    "DFS",
    "SPX",
    "EON",
    "EDF",
    "RWE",
    "SZR",
    "ICN",
    "CEIX",
    "MXEA",
    "SQ",
    "SHELL",
    "K",
}

MANUAL_YF_NAMES = {
    "^GSPC": "SP500_Price",
    "^IXIC": "NASDAQ_Price",
    "^DJI": "DOW_Price",
    "^RUT": "Russell_Price",
    "^VIX": "VIX_Price",
    "^VXN": "VXN_NASDAQ_Vol",
    "^OVX": "OVX_Oil_Vol",
    "^GVZ": "GVZ_Gold_Vol",
    "^EVZ": "EVZ_EUR_Vol",
    "^FTSE": "FTSE_UK",
    "^N225": "Nikkei_Japan",
    "^HSI": "HangSeng_HK",
    "^GDAXI": "DAX_Germany",
    "^FCHI": "CAC40_France",
    "^STOXX50E": "STOXX50E_EU",
    "SPY": "SPY",
    "QQQ": "QQQ",
    "TLT": "TLT_LongBond",
    "GLD": "GLD_Gold",
    "USO": "USO_Oil",
    "UUP": "UUP_Dollar",
    "FXE": "FXE_Euro",
    "FXY": "FXY_Yen",
    "HYG": "HYG_HighYield",
    "LQD": "LQD_InvGrade",

    # --- Tickers ajoutés (issus du dictionnaire massif fourni) ---
    "^BVSP": "BOVESPA_Brazil",
    "^AXJO": "ASX_Australia",
    "^AORD": "AORD_AUS",
    "^IBEX": "IBEX_Spain",
    "VXX": "VXX",
    "UVXY": "UVXY",
    "VIXY": "VIXY",
    "SVXY": "SVXY",
    "VXZ": "VXZ",
    "VIXM": "VIXM",
    "XLK": "XLK_Tech",
    "XLF": "XLF_Fin",
    "XLE": "XLE_Energy",
    "XLV": "XLV_Health",
    "XLU": "XLU_Util",
    "XLP": "XLP_Staples",
    "XLI": "XLI_Indust",
    "XLY": "XLY_Disc",
    "XLRE": "XLRE_RE",
    "XLB": "XLB_Materials",
    "XLC": "XLC_CommServ",
    "GOOGL": "GOOGL_Google",
    "META": "META_Meta",
    "AVGO": "AVGO_Broadcom",
    "ASML": "ASML_ASML",
    "WFC": "WFC_WellsFargo",
    "GS": "GS_GoldmanSachs",
    "BLK": "BLK_BlackRock",
    "SCHW": "SCHW_Schwab",
    "MS": "MS_MorganStanley",
    "COF": "COF_CapitalOne",
    "BAX": "BAX_BankBoston",
    "AXP": "AXP_Amex",
    "EQR": "EQR_Equity",
    "PLD": "PLD_Prologis",
    "AMT": "AMT_AmericanTower",
    "EQIX": "EQIX_Equinix",
    "CCI": "CCI_CrownCastle",
    "PSA": "PSA_PublicStorage",
    "ABBV": "ABBV_AbbVie",
    "MRK": "MRK_Merck",
    "BMY": "BMY_BristolMyers",
    "AMGN": "AMGN_Amgen",
    "GILD": "GILD_Gilead",
    "BNTX": "BNTX_BioNTech",
    "MRNA": "MRNA_Moderna",
    "CRSP": "CRSP_CrisprTherapy",
    "VRTX": "VRTX_VertexPharm",
    "ILMN": "ILMN_Illumina",
    "DXCM": "DXCM_Dexcom",
    "TDOC": "TDOC_Teladoc",
    "CI": "CI_Cigna",
    "HUM": "HUM_Humana",
    "RTX": "RTX_Raytheon",
    "LMT": "LMT_LockheedMartin",
    "NOC": "NOC_Northrop",
    "GD": "GD_GeneralDynamics",
    "CAT": "CAT_Caterpillar",
    "DE": "DE_Deere",
    "ITT": "ITT_ITTInc",
    "PAYX": "PAYX_Paychex",
    "CTAS": "CTAS_Cintas",
    "MMM": "3M",
    "HON": "HON_Honeywell",
    "ETN": "ETN_Eaton",
    "EMR": "EMR_Emerson",
    "OTIS": "OTIS_Otis",
    "JCI": "JCI_JohnsonControls",
    "PTC": "PTC_PTC",
    "SMCI": "SMCI_SuperMicroComputer",
    "COP": "COP_ConocoPhillips",
    "SLB": "SLB_Schlumberger",
    "EOG": "EOG_EOGResources",
    "MPC": "MPC_MarathonPetroleum",
    "PSX": "PSX_PhillipsLiquids",
    "VLO": "VLO_Valero",
    "PM": "PM_PhilipMorris",
    "MO": "MO_AltriaMG",
    "BTI": "BTI_BritishAmerican",
    "TBP": "TBP_Tata",
    "BP": "BP_BritishPetroleum",
    "TTE": "TTE_TotalEnergies",
    "ENB": "ENB_EnbridgeInc",
    "MET": "MET_MetalexEnergy",
    "ADM": "ADM_ArcherDaniels",
    "MKC": "MKC_McCormick",
    "SJM": "SJM_JM_Smucker",
    "CPB": "CPB_CampbellSoup",
    "GIS": "GIS_GeneralMills",
    "MDLZ": "MDLZ_Mondelez",
    "NSRGY": "NSRGY_Nestle",
    "TAP": "TAP_MolsonCoors",
    "BDX": "BDX_Becton_Dickinson",
    "CLX": "CLX_Clorox",
    "CL": "CL_Colgate",
    "UL": "UL_Unilever",
    "LVRK": "LVRK_Lavazza",
    "YUM": "YUM_YumBrands",
    "QSR": "QSR_RestaurantBrands",
    "DPZ": "DPZ_Dominos",
    "BLMN": "BLMN_BloombergME",
    "NWL": "NWL_Newell",
    "RRR": "RRR_RareMedica",
    "DASH": "DASH_DoorDash",
    "LYFT": "LYFT_Lyft",
    "UBER": "UBER_Uber",
    "TGT": "TGT_Target",
    "M": "M_Macys",
    "LOW": "LOW_Lowes",
    "ROST": "ROST_RossStores",
    "BBY": "BBY_BestBuy",
    "NEE": "NEE_NextEra",
    "DUK": "DUK_Duke",
    "SO": "SO_SouthernCo",
    "AEP": "AEP_AmericanElectric",
    "EXC": "EXC_Exelon",
    "SRE": "SRE_Sempra",
    "ES": "ES_Evergy",
    "XEL": "XEL_Xcel",
    "PPL": "PPL_PPL",
    "TMUS": "TMUS_TMobileUS",
    "CHTR": "CHTR_Charter",
    "VOD": "VOD_Vodafone",
    "TM": "TM_Telephone",
    "LOGI": "LOGI_Logitech",
    "NET": "NET_Cloudflare",
    "DDOG": "DDOG_Datadog",
    "SLV": "SLV_Silver",
    "UNG": "UNG_Gas",
    "DBC": "DBC_Commodity",
    "DBA": "DBA_Agri",
    "GDX": "GDX_GoldMiners",
    "GDXJ": "GDXJ_JrMiners",
    "PDBC": "PDBC_Commodity2",
    "CORN": "CORN_Corn",
    "SOYB": "SOYB_Soybean",
    "CBOT_W": "Wheat",
    "IEF": "IEF_MidBond",
    "SHY": "SHY_ShortBond",
    "SHV": "SHV_TBill",
    "BIL": "BIL_TBill3M",
    "AGG": "AGG_Aggregate",
    "BND": "BND_TotalBond",
    "JNK": "JNK_HY2",
    "VCIT": "VCIT_CorpIG",
    "VCSH": "VCSH_CorpST",
    "EMB": "EMB_EM",
    "MBB": "MBB_Mortgage",
    "TIP": "TIP_TIPS",
    "BNDX": "BNDX_IntlBond",
    "HYLD": "HYLD_HYieldETF",
    "PFFA": "PFFA_PreferredA",
    "EWJ": "EWJ_Japan",
    "EWG": "EWG_Germany",
    "EWU": "EWU_UK",
    "EWA": "EWA_Australia",
    "EWH": "EWH_HongKong",
    "EWL": "EWL_Switzerland",
    "EWP": "EWP_Spain",
    "EWI": "EWI_Italy",
    "EWQ": "EWQ_France",
    "EWT": "EWT_Taiwan",
    "EWY": "EWY_Korea",
    "EWZ": "EWZ_Brazil",
    "EWC": "EWC_Canada",
    "EWS": "EWS_Singapore",
    "EWM": "EWM_Malaysia",
    "FXI": "FXI_China",
    "MCHI": "MCHI_China2",
    "IEMG": "IEMG_EM",
    "EEM": "EEM_EM2",
    "VEA": "VEA_DM",
    "INDA": "INDA_India",
    "EPI": "EPI_India2",
    "ASHR": "ASHR_China_A",
    "TUR": "TUR_Turkey",
    "EIDO": "EIDO_Indonesia",
    "EPOL": "EPOL_Poland",
    "EZA": "EZA_SouthAfrica",
    "GXG": "GXG_Germany2",
    "EGRX": "EGRX_Greece",
    "FXB": "FXB_GBP",
    "FXA": "FXA_AUD",
    "FXC": "FXC_CAD",
    "FXF": "FXF_CHF",
    "CEW": "CEW_EM_FX",
    "CYB": "CYB_ChineseYuan",
    "BZF": "BZF_BrazilReal",
    "FXD": "FXD_SwedishKrona",
    "FXN": "FXN_NorwegianKrone",
    "GBTC": "GBTC_Bitcoin",
    "IBIT": "IBIT_Bitcoin2",
    "COIN": "COIN_Crypto",
    "MSTR": "MSTR_Bitcoin3",
    "BITO": "BITO_BitcoinETF",
    "ETHA": "ETHA_EthereumETF",
    "MARA": "MARA_Marathon",
    "RIOT": "RIOT_Riot",
    "CLSK": "CLSK_CleanSpark",
    "CIFR": "CIFR_Cipher",
    "CORZ": "CORZ_Core_Sci",
    "IWM": "IWM_SmallCap",
    "IVV": "IVV_SP500",
    "VTI": "VTI_Total",
    "VOO": "VOO_SP500_2",
    "VV": "VV_LargeCap",
    "VTV": "VTV_Value",
    "VUG": "VUG_Growth",
    "VB": "VB_SmallCap2",
    "SCHD": "SCHD_Div",
    "VIG": "VIG_DivGrowth",
    "HDV": "HDV_HighDiv",
    "NOBL": "NOBL_Aristocrat",
    "DGRO": "DGRO_DividendGrowth",
    "QUAL": "QUAL_Quality",
    "VLUE": "VLUE_Value",
    "VYMI": "VYMI_HighDivYield",
    "JEPI": "JEPI_EquityPremiumIncome",
    "XYLD": "XYLD_XYieldETF",
    "QYLD": "QYLD_NasdaqYield",
    "RYLD": "RYLD_Russell2000Yield",
    "ARKK": "ARKK_Innovation",
    "XBI": "XBI_Biotech",
    "SOXX": "SOXX_Semis",
    "IBB": "IBB_Biotech2",
    "IYT": "IYT_Transport",
    "XHB": "XHB_Homebuilders",
    "KRE": "KRE_RegionalBanks",
    "KBE": "KBE_Banks",
    "ITA": "ITA_Defense",
    "XOP": "XOP_OilExploration",
    "OIH": "OIH_OilServices",
    "IYM": "IYM_BasicMaterials",
    "PCAR": "PCAR_PaccarInc",
    "DAL": "DAL_Delta",
    "AAL": "AAL_AmericanAir",
    "UAL": "UAL_UnitedAir",
    "LUV": "LUV_SouthwestAir",
    "ICLN": "ICLN_CleanEnergy",
    "TAN": "TAN_SolarEnergy",
    "MTUM": "MTUM_Momentum",
    "USMV": "USMV_MinVol",
    "SPLV": "SPLV_LowVol_SP500",
    "RSP": "RSP_EqualWeight_SP500",
    "EUSA": "EUSA_EuropeMomentum",
    "EEMV": "EEMV_EMMinVol",
    "VNQ": "VNQ_US_REIT",
    "IYR": "IYR_US_REIT2",
    "REM": "REM_Mortgage_REIT",
    "SPG": "SPG_SimonProperty",
    "AVB": "AVB_AvalonBay",
    "COLD": "COLD_ColdStorage",
    "DLR": "DLR_Digital_Realty",
    "REXR": "REXR_Rexford",
    "HII": "HII_HuntingtonIngalls",
    "L3HARRIS": "L3H_L3Harris",
    "LDOS": "LDOS_LeadosSecurity",
    "EBAY": "EBAY_eBay",
    "MELI": "MELI_MercadoLibre",
    "SHOP": "SHOP_Shopify",
    "SE": "SE_SeaLimited",
    "PDD": "PDD_PinDuoDuo",
    "JD": "JD_JD.com",
    "VIPS": "VIPS_Vipshop",
    "UPST": "UPST_Upstart",
    "RBLX": "RBLX_Roblox",
    "SNOW": "SNOW_Snowflake",
    "CRWD": "CRWD_CrowdStrike",
    "ZM": "ZM_Zoom",
    "ROKU": "ROKU_Roku",
    "PINS": "PINS_Pinterest",
    "SNAP": "SNAP_Snapchat",
    "TERM": "TERM_Terminal",
    "SPCE": "SPCE_VirginGalactic",
}

YF_TICKERS_RAW = """
    ^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ
    ^FTSE ^N225 ^HSI ^GDAXI ^FCHI ^STOXX50E
    SPY QQQ TLT GLD USO UUP FXE FXY HYG LQD
    AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA

    ^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF XLE XLV XLU XLP XLI XLY XLRE
    XLB XLC GOOGL META AVGO ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT EQIX CCI PSA ABBV
    MRK BMY AMGN GILD BNTX MRNA CRSP VRTX ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
    PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB EOG MPC PSX VLO PM MO BTI TBP BP TTE
    ENB MET ADM MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL LVRK YUM QSR DPZ BLMN NWL RRR
    DASH LYFT UBER TGT M LOW ROST BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR VOD TM LOGI
    NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC CORN SOYB CBOT_W IEF SHY SHV BIL AGG BND JNK VCIT
    VCSH EMB MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP EWI EWQ EWT EWY EWZ EWC EWS
    EWM FXI MCHI IEMG EEM VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA FXC FXF CEW
    CYB BZF FXD FXN GBTC IBIT COIN MSTR BITO ETHA MARA RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV
    VTV VUG VB SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD ARKK XBI SOXX IBB
    IYT XHB KRE KBE ITA XOP OIH IYM PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
    EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII L3HARRIS LDOS EBAY MELI SHOP SE PDD JD VIPS
    UPST RBLX SNOW CRWD ZM ROKU PINS SNAP TERM SPCE
"""

def sanitize_name(ticker: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", ticker.replace("^", "IDX_"))
    return name.strip("_")


def build_yf_dict() -> Dict[str, str]:
    tickers = []

    for t in YF_TICKERS_RAW.split():
        t = t.strip()
        if not t:
            continue
        if t in BAD_TICKERS:
            continue
        tickers.append(t)

    seen = set()
    unique_tickers = []

    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    yf_dict = {}
    used_names = set()

    for ticker in unique_tickers:
        base_name = MANUAL_YF_NAMES.get(ticker, sanitize_name(ticker))
        name = base_name
        i = 2

        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1

        used_names.add(name)
        yf_dict[ticker] = name

    return yf_dict


# =============================================================================
# FRED INDICATORS
# =============================================================================

fred_dict = {
    "VIXCLS": "VIX",
    "VIXDVOL": "VIX_DrawVol",
    "OILPRICE": "Oil_Price",
    "SP500": "SP500_Level",
    "WILL5000IND": "Wilshire5000",

    "DCOILWTICO": "WTI_Oil_FRED",
    "DCOILBRENTEU": "Brent_Oil_FRED",

    "DGS30": "US30Y_Rate",
    "DGS20": "US20Y_Rate",
    "DGS10": "US10Y_Rate",
    "DGS7": "US7Y_Rate",
    "DGS5": "US5Y_Rate",
    "DGS3": "US3Y_Rate",
    "DGS2": "US2Y_Rate",
    "DGS1": "US1Y_Rate",
    "DTB6": "US6M_Rate",
    "DTB3": "US3M_Rate",
    "DTB1": "US1M_Rate",

    "FEDFUNDS": "FedFunds",
    "EFFR": "EFFR",
    "SOFR": "SOFR_SecuredOIS",
    "DFF": "DFF",

    "T10Y2Y": "T10Y2Y_Spread",
    "T10Y3M": "T10Y3M_Spread",
    "T10YIE": "T10Y_Inflation_Expectation",
    "T5YIE": "T5Y_Inflation_Expectation",
    "T5YIFR": "T5Y5Y_Inflation_Forward",
    "TEDRATE": "TED_Spread",
    "BAMLH0A0HYM2": "HY_OAS",
    "BAMLC0A0CM": "IG_OAS",
    "BAMLC0A4CBBB": "BBB_OAS",

    "UNRATE": "Unemployment",
    "PAYEMS": "NonfarmPayrolls",
    "CPIAUCSL": "CPI",
    "CPILFESL": "Core_CPI",
    "PCE": "PCE",
    "PCEPILFE": "Core_PCE",
    "GDP": "GDP",
    "INDPRO": "Industrial_Production",
    "UMCSENT": "Michigan_Sentiment",
    "RSAFS": "Retail_Sales",

    "NFCI": "NFCI",
    "STLFSI4": "STLFSI4",
}

In [51]:
def safe_series(df: pd.DataFrame, col: str) -> pd.Series:

    x = df.loc[:, col]

    # si plusieurs colonnes (cas MultiIndex / duplicates)
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]

    # conversion safe
    x = pd.to_numeric(x, errors="coerce")

    # garantit alignement index
    x.index = df.index

    return x


def safe_bool_series(x: pd.Series) -> pd.Series:
    """
    Force une Series en bool propre.
    Évite le bug ~True = -2 / ~False = -1 si dtype=object.
    """
    return x.fillna(False).astype(bool)


In [52]:
# =============================================================================
# DATA LOADER
# =============================================================================

class DataLoader:
    def __init__(self):
        self.yf_failed = []
        self.fred_failed = []

    def load_yfinance_massive(self, tickers: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        all_tickers = list(tickers.keys())
        chunks = [
            all_tickers[i:i + YF_CHUNK_SIZE]
            for i in range(0, len(all_tickers), YF_CHUNK_SIZE)
        ]

        all_data = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"[DATA] Yahoo chunk {idx}/{len(chunks)} | tickers={len(chunk)}")

            try:
                raw = yf.download(
                    tickers=chunk,
                    start=start_date,
                    end=end_date,
                    progress=False,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True
                )

                if raw is None or raw.empty:
                    self.yf_failed.extend(chunk)
                    continue

                for ticker in chunk:
                    try:
                        col_name = tickers[ticker]

                        if isinstance(raw.columns, pd.MultiIndex):
                            if ticker not in raw.columns.get_level_values(0):
                                self.yf_failed.append(ticker)
                                continue

                            ticker_df = raw[ticker]

                            if "Close" not in ticker_df.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = ticker_df["Close"]

                        else:
                            if len(chunk) != 1 or "Close" not in raw.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = raw["Close"]

                        close = pd.to_numeric(close, errors="coerce")
                        close = close.rename(col_name).to_frame()
                        close.index = pd.to_datetime(close.index)
                        close = close[~close.index.duplicated(keep="last")]
                        close = close.sort_index()

                        if close.dropna().shape[0] >= 100:
                            all_data.append(close)
                        else:
                            self.yf_failed.append(ticker)

                    except Exception:
                        self.yf_failed.append(ticker)

            except Exception:
                self.yf_failed.extend(chunk)

            time.sleep(SLEEP_BETWEEN_CHUNKS)

        if not all_data:
            return pd.DataFrame()

        df = pd.concat(all_data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def load_fred(self, indicators: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        data = []

        for i, (code, col_name) in enumerate(indicators.items(), 1):
            print(f"[DATA] FRED {i}/{len(indicators)} | {code}")

            try:
                raw = pdr.get_data_fred(code, start=start_date, end=end_date)

                if raw is None or raw.empty:
                    self.fred_failed.append(code)
                    continue

                series = raw.iloc[:, 0]
                series = pd.to_numeric(series, errors="coerce")
                series = series.rename(col_name).to_frame()
                series.index = pd.to_datetime(series.index)
                series = series[~series.index.duplicated(keep="last")]
                series = series.sort_index()

                if series.dropna().shape[0] >= 30:
                    data.append(series)
                else:
                    self.fred_failed.append(code)

            except Exception:
                self.fred_failed.append(code)

            time.sleep(0.1)

        if not data:
            return pd.DataFrame()

        df = pd.concat(data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def combine_to_latest_full_dataset(self, yf_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
        """
        CORRECTIF (bug identifié) : l'ancienne version calculait la couverture
        de chaque colonne (notna().mean()) puis filtrait les LIGNES à >=95% de
        couverture. Comme plusieurs colonnes (tickers/indices créés après 2000,
        ex: OVX 2007, GVZ 2008) n'ont pas d'historique avant ~2010, la
        couverture moyenne des lignes < 2010 tombait sous le seuil et TOUTES
        les lignes pré-2010 étaient supprimées - peu importe MIN_DATA_START.
        Conséquence concrète : tout le sweep TRAIN_START=2000..2010 utilisait
        en réalité toujours la même fenêtre (~2010+), donc les 22 runs du
        sweep étaient quasi identiques entre eux.

        CORRECTIF appliqué : on calcule la couverture PAR COLONNE sur toute la
        fenêtre demandée (depuis MIN_DATA_START), et on retire en amont les
        colonnes dont la couverture est insuffisante - PAS les lignes. Une
        colonne sans historique avant 2010 est donc exclue du feature set
        pour ce run, mais les lignes 2000-2009 sont conservées avec les
        features qui, elles, couvrent bien toute la période.
        """
        if yf_df.empty:
            raise ValueError("Yahoo Finance data is empty.")

        yf_df = yf_df.sort_index()
        yf_df.index = pd.to_datetime(yf_df.index)
        yf_df = yf_df.loc[yf_df.index >= MIN_DATA_START].copy()
        yf_df = yf_df.dropna(axis=1, how="all")

        if yf_df.empty:
            raise ValueError("Yahoo Finance dataframe has no usable columns since MIN_DATA_START.")

        if not fred_df.empty:
            fred_df = fred_df.sort_index()
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df = fred_df.loc[fred_df.index >= MIN_DATA_START].copy()
            fred_df = fred_df.dropna(axis=1, how="all")

            fred_on_market_calendar = fred_df.reindex(yf_df.index).ffill()
            combined = pd.concat([yf_df, fred_on_market_calendar], axis=1)
        else:
            combined = yf_df.copy()

        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined = combined.replace([np.inf, -np.inf], np.nan)
        combined = combined.sort_index()

        if "VIX_Price" not in combined.columns and "VIX" not in combined.columns:
            raise ValueError("No usable VIX column found after combining data.")

        # --- Filtre 1 : colonne apparue trop tard après MIN_DATA_START ---
        latest_allowed_first_valid = MIN_DATA_START + pd.Timedelta(days=MAX_FIRST_VALID_LAG_DAYS)

        keep_cols_start = []
        for col in combined.columns:
            first_valid = combined[col].first_valid_index()
            if first_valid is None:
                continue
            if first_valid <= latest_allowed_first_valid:
                keep_cols_start.append(col)

        combined = combined[keep_cols_start]

        if combined.empty:
            raise ValueError("No columns left after first-valid-date filter.")

        combined = combined.ffill()

        # --- Filtre 2 (CORRIGÉ) : couverture par COLONNE sur toute la fenêtre,
        # pas de filtre de ligne. Seuil MIN_COLUMN_COVERAGE (ex: 0.90). ---
        coverage = combined.notna().mean()
        keep_cols_coverage = coverage[coverage >= MIN_COLUMN_COVERAGE].index.tolist()

        core_cols = [
            "VIX_Price",
            "VIX",
            "SP500_Price",
            "SPY",
            "QQQ",
            "TLT_LongBond",
            "GLD_Gold",
            "USO_Oil"
        ]

        for c in core_cols:
            if c in combined.columns and c not in keep_cols_coverage:
                if combined[c].notna().mean() >= 0.75:
                    keep_cols_coverage.append(c)

        n_cols_before = combined.shape[1]
        combined = combined[keep_cols_coverage]
        print(f"[COMBINE] Colonnes retirées par couverture insuffisante (<{MIN_COLUMN_COVERAGE:.0%}): "
              f"{n_cols_before - combined.shape[1]}/{n_cols_before}")

        if combined.empty:
            raise ValueError("No columns left after coverage filter.")

        # --- PAS de filtre de ligne (row_coverage) ici : on garde toutes les
        # lignes depuis MIN_DATA_START, puisque les colonnes retenues couvrent
        # déjà >= MIN_COLUMN_COVERAGE de cette fenêtre par construction. ---
        combined = combined.ffill()
        combined = combined.dropna(axis=0, how="any")

        if combined.empty:
            raise ValueError("Final dataset is empty after coverage filters.")

        print(f"[COMBINE] Columns kept: {combined.shape[1]}")
        print(f"[COMBINE] Rows kept:    {combined.shape[0]}")
        print(f"[COMBINE] Date range:   {combined.index.min().date()} → {combined.index.max().date()}")

        return combined


In [53]:
class FeatureEngineer:
    def __init__(self):
        self.features = []

    def add(self, name: str):
        if name not in self.features:
            self.features.append(name)

    def create_features(self, df: pd.DataFrame):
        print("[FEATURES] Creating features...")

        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]
        base_cols = list(df.columns)

        for col in base_cols:
            s = safe_series(df, col)
            # Replace 0 values with NaN to avoid ZeroDivisionError in pct_change
            s_clean = s.replace(0, np.nan)

            # returns
            for p in [1, 5, 20]:
                f = f"{col}_ret_{p}d"
                df[f] = s_clean.pct_change(p)
                self.add(f)

            # volatility
            f = f"{col}_vol_20d"
            df[f] = s_clean.pct_change().rolling(20).std()
            self.add(f)

            # z-score
            mean = s_clean.rolling(60).mean()
            std = s_clean.rolling(60).std()
            f = f"{col}_zscore_60d"
            df[f] = (s_clean - mean) / (std + 1e-8)
            self.add(f)

        # VIX features
        vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX" if "VIX" in df.columns else None

        if vix_col:
            vix = safe_series(df, vix_col)

            df["vix_level"] = vix
            df["vix_change_1d"] = vix.pct_change(1)
            df["vix_change_5d"] = vix.pct_change(5)
            df["vix_ma_20"] = vix.rolling(20).mean()
            df["vix_vs_ma20"] = vix - df["vix_ma_20"]

            for f in [
                "vix_level",
                "vix_change_1d",
                "vix_change_5d",
                "vix_ma_20",
                "vix_vs_ma20"
            ]:
                self.add(f)

        # SP500 features
        if "SP500_Price" in df.columns:
            spx = safe_series(df, "SP500_Price")

            df["spx_realized_vol_20d"] = spx.pct_change().rolling(20).std() * np.sqrt(252)

            # Fix for spx_drawdown_252d to prevent data leakage:
            # Calculate the peak from the *previous* 252 days, excluding the current day.
            # This ensures the feature for day 't' only uses data available up to day 't-1'.
            spx_peak_before_today = spx.rolling(window=252, closed='left').max()
            df["spx_drawdown_252d"] = (spx_peak_before_today - spx) / (spx_peak_before_today + 1e-8)
            df["spx_drawdown_252d"] = df["spx_drawdown_252d"].clip(lower=0) # Drawdown cannot be negative

            df["spx_down_day"] = (spx.pct_change(1) < 0).astype(int)

            for f in [
                "spx_realized_vol_20d",
                "spx_drawdown_252d",
                "spx_down_day"
            ]:
                self.add(f)

        # Yield curve
        if "US10Y_Rate" in df.columns and "US2Y_Rate" in df.columns:
            us10 = safe_series(df, "US10Y_Rate")
            us2 = safe_series(df, "US2Y_Rate")

            df["yield_curve_10y_2y"] = us10 - us2
            df["yield_curve_change_20d"] = df["yield_curve_10y_2y"].diff(20)

            for f in [
                "yield_curve_10y_2y",
                "yield_curve_change_20d"
            ]:
                self.add(f)

        self.features = [f for f in self.features if f in df.columns]

        return df, self.features

In [54]:
class TargetBuilder:
    """
    Construit la cible (VIX_Direction) et le régime de marché (VIX_Regime).

    horizon_days controle l'horizon de prediction : la cible devient
    "le VIX monte-t-il entre T et T+horizon_days" au lieu de T et T+1 fixe.
    Avec horizon_days=1 (defaut), comportement strictement identique a avant.

    Deux modes pour définir les seuils de régime CALM/NORMAL/STRESS :

    - mode="fixed"   : quantiles q33/q67 calculés UNE FOIS sur la période
                        train (< train_end), puis appliqués tels quels à
                        tout le dataframe (train + test). Pas de fuite
                        train->test, mais le seuil ne s'adapte pas si le
                        régime de volatilité change structurellement avec
                        le temps (ex: VIX 2008 vs VIX 2017).

    - mode="rolling" : quantiles q33/q67 recalculés à CHAQUE date T sur une
                        fenêtre glissante des `rolling_window` jours
                        précédents (closed='left', donc strictement avant T
                        - pas de fuite intra-jour). Le régime à la date T
                        reflète le niveau de VIX relatif à son contexte
                        récent (~2 ans avec rolling_window=504), pas à toute
                        l'histoire 2000-2026 mélangée.
    """
    def __init__(self, q_low: float = 0.33, q_high: float = 0.67,
                 mode: str = "fixed", rolling_window: int = 504,
                 horizon_days: int = 1):
        assert mode in ("fixed", "rolling"), "mode must be 'fixed' or 'rolling'"
        assert horizon_days >= 1, "horizon_days must be >= 1"
        self.vix_col = None
        self.q_low = q_low
        self.q_high = q_high
        self.mode = mode
        self.rolling_window = rolling_window
        self.horizon_days = horizon_days  # horizon de prediction en jours de bourse (1, 3, 5...)
        self.calm_threshold_ = None    # scalar if mode="fixed", else None
        self.stress_threshold_ = None
        self.flat_threshold = VIX_FLAT_PCT_THRESHOLD
        self.flat_removed = 0
        self.total_before_flat_filter = 0

    def build(self, df: pd.DataFrame, train_end: str = None) -> pd.DataFrame:
        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]

        if "VIX_Price" in df.columns:
            self.vix_col = "VIX_Price"
        elif "VIX" in df.columns:
            self.vix_col = "VIX"
        else:
            raise KeyError("No VIX column found.")

        vix = safe_series(df, self.vix_col)
        # Horizon de prediction : shift(-horizon_days) au lieu de shift(-1) fixe.
        # horizon_days=1 reproduit exactement le comportement d'origine.
        future_vix = vix.shift(-self.horizon_days)

        vix_next_change = (future_vix / vix) - 1

        df["VIX_Next_Change"] = vix_next_change
        df["VIX_Is_Flat"] = vix_next_change.abs() <= self.flat_threshold

        direction = pd.Series(np.nan, index=df.index)
        direction.loc[vix_next_change > 0] = 1
        direction.loc[vix_next_change <= 0] = 0
        direction.loc[future_vix.isna()] = np.nan

        df["VIX_Direction"] = direction
        df.loc[future_vix.isna(), "VIX_Is_Flat"] = np.nan

        if self.mode == "fixed":
            # --- Quantiles fixes, calculés sur le train seulement (no leakage) ---
            if train_end is not None:
                vix_for_quantiles = vix.loc[vix.index < pd.Timestamp(train_end)]
            else:
                vix_for_quantiles = vix

            self.calm_threshold_ = vix_for_quantiles.quantile(self.q_low)
            self.stress_threshold_ = vix_for_quantiles.quantile(self.q_high)

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < self.calm_threshold_] = "CALM"
            regime.loc[vix >= self.stress_threshold_] = "STRESS"

            threshold_desc = (
                f"CALM < {self.calm_threshold_:.2f}, "
                f"NORMAL [{self.calm_threshold_:.2f}-{self.stress_threshold_:.2f}), "
                f"STRESS >= {self.stress_threshold_:.2f}  (fixe, calculé sur train)"
            )

        else:
            # --- Quantiles rolling : recalculés à chaque date T sur les
            # `rolling_window` jours STRICTEMENT précédents (closed='left').
            # Pas de fuite : le quantile à T n'utilise jamais VIX(T) ni le futur.
            rolling_calm = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_low)
            rolling_stress = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_high)

            self.calm_threshold_ = rolling_calm   # Series, pas un scalaire
            self.stress_threshold_ = rolling_stress

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < rolling_calm] = "CALM"
            regime.loc[vix >= rolling_stress] = "STRESS"
            # Tant que la fenêtre rolling n'est pas pleine (début d'historique),
            # rolling_calm/rolling_stress sont NaN -> régime indéfini -> ces
            # lignes seront retirées plus bas (dropna sur VIX_Regime_valid).
            regime.loc[rolling_calm.isna() | rolling_stress.isna()] = np.nan

            threshold_desc = (
                f"rolling sur {self.rolling_window} jours (~{self.rolling_window/252:.1f} ans), "
                f"recalculé à chaque date T sur les jours STRICTEMENT antérieurs à T"
            )

        df["VIX_Regime"] = regime

        # Lignes à retirer : direction NaN (fin de série), OU régime NaN (mode
        # rolling, début de série sans assez d'historique pour la fenêtre)
        df = df.dropna(subset=["VIX_Direction", "VIX_Is_Flat", "VIX_Regime"]).copy()

        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].fillna(False).astype(bool)

        self.total_before_flat_filter = int(len(df))
        self.flat_removed = int(df["VIX_Is_Flat"].sum())

        df = df.loc[~df["VIX_Is_Flat"]].copy()

        df["VIX_Direction"] = df["VIX_Direction"].astype(int)
        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].astype(bool)

        print(f"[TARGET] VIX column: {self.vix_col}  |  mode={self.mode}")
        print(f"[TARGET] VIX regime thresholds: {threshold_desc}")
        print(f"[TARGET] VIX flat threshold: ±{self.flat_threshold:.2%}")
        print(f"[TARGET] Flat days removed completely: {self.flat_removed}/{self.total_before_flat_filter}")
        print(f"[TARGET] Remaining non-flat rows: {len(df)}")
        print(df["VIX_Regime"].value_counts())

        return df



In [55]:
class TrainFittedCleaner:
    def __init__(self):
        self.medians = None
        self.lower = None
        self.upper = None

    def fit(self, X: pd.DataFrame):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.medians = X.median()
        self.lower = X.quantile(0.01)
        self.upper = X.quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians)
        X = X.clip(lower=self.lower, upper=self.upper, axis=1)
        X = X.fillna(0)
        return X

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self.fit(X)
        return self.transform(X)

In [56]:
def evaluate_model_across_regimes(df_test, trained_by_regime):
    all_preds = pd.Series(index=df_test.index, dtype=float)
    all_probas = pd.Series(index=df_test.index, dtype=float)

    for regime, pack in trained_by_regime.items():
        mask = df_test["VIX_Regime"] == regime

        if mask.sum() == 0:
            continue

        X_raw = df_test.loc[mask, pack["features"]]
        X_clean = pack["cleaner"].transform(X_raw)
        X_scaled = pack["scaler"].transform(X_clean)

        model = pack["model"]

        pred = model.predict(X_scaled)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_scaled)[:, 1]
        else:
            proba = pred.astype(float)

        all_preds.loc[mask] = pred
        all_probas.loc[mask] = proba

    valid = all_preds.notna()

    y_true = df_test.loc[valid, "VIX_Direction"].values
    y_pred = all_preds.loc[valid].astype(int).values
    y_proba = all_probas.loc[valid].values

    if len(y_true) == 0:
        return None

    return compute_metrics(y_true, y_pred, y_proba)

In [57]:
def model_configs():
    return {
        "XGBoost": (
            XGBClassifier,
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "subsample": [0.8],
                "colsample_bytree": [0.8]
            },
            {
                "random_state": RANDOM_STATE,
                "eval_metric": "logloss",
                "n_jobs": -1
            }
        ),
        "LightGBM": (
            LGBMClassifier,
            {
                "num_leaves": [7, 15],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "max_depth": [3, 5]
            },
            {
                "random_state": RANDOM_STATE,
                "verbose": -1,
                "class_weight": "balanced"
            }
        ),
        "GradientBoosting": (
            GradientBoostingClassifier,
            {
                "n_estimators": [75, 125],
                "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3],
                "min_samples_leaf": [10]
            },
            {
                "random_state": RANDOM_STATE
            }
        ),
        "RandomForest": (
            RandomForestClassifier,
            {
                "n_estimators": [150],
                "max_depth": [3, 5],
                "min_samples_leaf": [10, 20]
            },
            {
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "class_weight": "balanced"
            }
        ),
        "LogisticRegression": (
            LogisticRegression,
            {
                "C": [0.01, 0.1, 1.0],
                "penalty": ["l2"]
            },
            {
                "random_state": RANDOM_STATE,
                "max_iter": 2000,
                "class_weight": "balanced"
            }
        ),
    }


In [58]:
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "R2": r2_score(y_true, y_proba),
        "AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Pred_0": int((y_pred == 0).sum()),
        "Pred_1": int((y_pred == 1).sum()),
        "Actual_0": int((y_true == 0).sum()),
        "Actual_1": int((y_true == 1).sum()),
        "Confusion_Matrix": cm.tolist()
    }

In [59]:
print("[INIT] Building ticker dictionaries...")
yf_dict = build_yf_dict()
print(f"[INIT] Yahoo tickers: {len(yf_dict)}")
print(f"[INIT] FRED indicators: {len(fred_dict)}")

[INIT] Building ticker dictionaries...
[INIT] Yahoo tickers: 345
[INIT] FRED indicators: 43


In [62]:
loader = DataLoader()
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
print("[STEP 1/6] Loading data...")
yf_df = loader.load_yfinance_massive(yf_dict, TRAIN_START, end_date)
fred_df = loader.load_fred(fred_dict, TRAIN_START, end_date)

print("[STEP 2/6] Combining data...")
df = loader.combine_to_latest_full_dataset(yf_df, fred_df)


[STEP 1/6] Loading data...
[DATA] Yahoo chunk 1/9 | tickers=40
[DATA] Yahoo chunk 2/9 | tickers=40
[DATA] Yahoo chunk 3/9 | tickers=40
[DATA] Yahoo chunk 4/9 | tickers=40
[DATA] Yahoo chunk 5/9 | tickers=40


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LVRK"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LVRK']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 6/9 | tickers=40


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CBOT_W"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CBOT_W']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 7/9 | tickers=40
[DATA] Yahoo chunk 8/9 | tickers=40
[DATA] Yahoo chunk 9/9 | tickers=25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['L3HARRIS']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] FRED 1/43 | VIXCLS
[DATA] FRED 2/43 | VIXDVOL
[DATA] FRED 3/43 | OILPRICE
[DATA] FRED 4/43 | SP500
[DATA] FRED 5/43 | WILL5000IND
[DATA] FRED 6/43 | DCOILWTICO
[DATA] FRED 7/43 | DCOILBRENTEU
[DATA] FRED 8/43 | DGS30
[DATA] FRED 9/43 | DGS20
[DATA] FRED 10/43 | DGS10
[DATA] FRED 11/43 | DGS7
[DATA] FRED 12/43 | DGS5
[DATA] FRED 13/43 | DGS3
[DATA] FRED 14/43 | DGS2
[DATA] FRED 15/43 | DGS1
[DATA] FRED 16/43 | DTB6
[DATA] FRED 17/43 | DTB3
[DATA] FRED 18/43 | DTB1
[DATA] FRED 19/43 | FEDFUNDS
[DATA] FRED 20/43 | EFFR
[DATA] FRED 21/43 | SOFR
[DATA] FRED 22/43 | DFF
[DATA] FRED 23/43 | T10Y2Y
[DATA] FRED 24/43 | T10Y3M
[DATA] FRED 25/43 | T10YIE
[DATA] FRED 26/43 | T5YIE
[DATA] FRED 27/43 | T5YIFR
[DATA] FRED 28/43 | TEDRATE
[DATA] FRED 29/43 | BAMLH0A0HYM2
[DATA] FRED 30/43 | BAMLC0A0CM
[DATA] FRED 31/43 | BAMLC0A4CBBB
[DATA] FRED 32/43 | UNRATE
[DATA] FRED 33/43 | PAYEMS
[DATA] FRED 34/43 | CPIAUCSL
[DATA] FRED 35/43 | CPILFESL
[DATA] FRED 36/43 | PCE
[DATA] FRED 37/43 | PCEPILF

In [63]:
print("[STEP 3/6] Feature engineering...")
engineer = FeatureEngineer()
df, features = engineer.create_features(df)

[STEP 3/6] Feature engineering...
[FEATURES] Creating features...


In [64]:
# Copie de référence du dataframe après feature engineering, AVANT target/régime.
# Sert de point de départ identique pour les deux modes de quantile (fixed/rolling).
df_post_features = df.copy()
features_post_engineering = list(features)
print(f"[CHECKPOINT] df_post_features: {df_post_features.shape}, features: {len(features_post_engineering)}")


[CHECKPOINT] df_post_features: (6737, 1180), features: 985


In [65]:
print("[STEP] Chargement et feature engineering terminés. "
      "Démarrage de la Phase 1 (scoring des fenêtres par régime).")


[STEP] Chargement et feature engineering terminés. Démarrage de la Phase 1 (scoring des fenêtres par régime).


In [66]:
# =============================================================================
# FEATURES D'AMPLITUDE — calculées directement sur VIX et SPX
# Objectif : capturer l'ACCÉLÉRATION et l'ERRATICITÉ du VIX, pas seulement
# la direction. Ce sont les features manquantes pour prédire UP_FORT.
# =============================================================================

def add_amplitude_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ajoute des features spécifiques à l'intensité du mouvement VIX."""
    df = df.copy()

    # Colonne VIX brute
    vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX"
    spx_col = "SP500_Price" if "SP500_Price" in df.columns else None
    vix = df[vix_col]

    # 1. Volatilité-de-la-volatilité : vol réalisée du VIX sur 5 et 10 jours
    df["vix_vol_of_vol_5d"]  = vix.pct_change().rolling(5,  min_periods=3).std()
    df["vix_vol_of_vol_10d"] = vix.pct_change().rolling(10, min_periods=5).std()

    # 2. Momentum à court terme : accélération récente du VIX
    vix_ret1 = vix.pct_change(1)
    df["vix_momentum_2d"] = vix.pct_change(2)
    df["vix_momentum_3d"] = vix.pct_change(3)

    # 3. Accélération : changement de vitesse (dérivée seconde)
    df["vix_acceleration_1d"] = vix_ret1 - vix_ret1.shift(1)
    df["vix_acceleration_3d"] = vix_ret1 - vix_ret1.shift(3)

    # 4. Sur-extension court terme : distance au MA5 et MA10
    ma5  = vix.rolling(5,  min_periods=3).mean()
    ma10 = vix.rolling(10, min_periods=5).mean()
    df["vix_vs_ma5"]  = (vix - ma5)  / ma5.replace(0, np.nan)
    df["vix_vs_ma10"] = (vix - ma10) / ma10.replace(0, np.nan)

    # 5. Z-score court terme du VIX (5 et 10 jours)
    df["vix_zscore_5d"]  = (vix - ma5)  / vix.rolling(5,  min_periods=3).std().replace(0, np.nan)
    df["vix_zscore_10d"] = (vix - ma10) / vix.rolling(10, min_periods=5).std().replace(0, np.nan)

    # 6. Persistance du stress : % des 10 derniers jours avec VIX > MA20
    ma20 = vix.rolling(20, min_periods=10).mean()
    above_ma20 = (vix > ma20).astype(float)
    df["vix_pct_above_ma20_10d"] = above_ma20.rolling(10, min_periods=5).mean()

    # 7. Erraticité : max des |rendements| sur 5 jours vs moyenne
    abs_ret = vix.pct_change().abs()
    df["vix_max_abs_ret_5d"]  = abs_ret.rolling(5, min_periods=3).max()
    df["vix_mean_abs_ret_5d"] = abs_ret.rolling(5, min_periods=3).mean()
    df["vix_erratic_ratio"]   = df["vix_max_abs_ret_5d"] / df["vix_mean_abs_ret_5d"].replace(0, np.nan)

    # 8. Ratio vol_5d / vol_60d (spike court terme vs bruit de fond)
    df["vix_vol_ratio_5_60"] = (
        vix.pct_change().rolling(5,  min_periods=3).std() /
        vix.pct_change().rolling(60, min_periods=30).std().replace(0, np.nan)
    )

    # 9. SPX amplitude features (si disponible)
    if spx_col and spx_col in df.columns:
        spx = df[spx_col]
        spx_ret = spx.pct_change()
        df["spx_vol_5d"]         = spx_ret.rolling(5,  min_periods=3).std()
        df["spx_momentum_3d"]    = spx.pct_change(3)
        df["spx_abs_ret_max_5d"] = spx_ret.abs().rolling(5, min_periods=3).max()

    df = df.replace([np.inf, -np.inf], np.nan)
    n_added = sum(1 for c in df.columns if c in [
        "vix_vol_of_vol_5d","vix_vol_of_vol_10d","vix_momentum_2d","vix_momentum_3d",
        "vix_acceleration_1d","vix_acceleration_3d","vix_vs_ma5","vix_vs_ma10",
        "vix_zscore_5d","vix_zscore_10d","vix_pct_above_ma20_10d",
        "vix_max_abs_ret_5d","vix_mean_abs_ret_5d","vix_erratic_ratio",
        "vix_vol_ratio_5_60","spx_vol_5d","spx_momentum_3d","spx_abs_ret_max_5d"
    ])
    print(f"[AMPLITUDE FEATURES] {n_added} features d'amplitude ajoutées")
    return df

df_post_features = add_amplitude_features(df_post_features)

# Mettre à jour la liste de features
amplitude_feat_names = [
    "vix_vol_of_vol_5d","vix_vol_of_vol_10d","vix_momentum_2d","vix_momentum_3d",
    "vix_acceleration_1d","vix_acceleration_3d","vix_vs_ma5","vix_vs_ma10",
    "vix_zscore_5d","vix_zscore_10d","vix_pct_above_ma20_10d",
    "vix_max_abs_ret_5d","vix_mean_abs_ret_5d","vix_erratic_ratio",
    "vix_vol_ratio_5_60","spx_vol_5d","spx_momentum_3d","spx_abs_ret_max_5d"
]
new_feats = [f for f in amplitude_feat_names if f in df_post_features.columns]
features_post_engineering = features_post_engineering + new_feats
print(f"[AMPLITUDE FEATURES] features_post_engineering : {len(features_post_engineering)} total ({len(new_feats)} nouvelles)")


[AMPLITUDE FEATURES] 18 features d'amplitude ajoutées
[AMPLITUDE FEATURES] features_post_engineering : 1003 total (18 nouvelles)


In [67]:
# =============================================================================
# PRÉPARATION DES SÉRIES — VIX et features communes aux 4 modèles
# =============================================================================

vix_col = "VIX_Price" if "VIX_Price" in df_post_features.columns else "VIX"
spx_col = "SP500_Price" if "SP500_Price" in df_post_features.columns else None

vix = df_post_features[vix_col].dropna().sort_index()
vix_ret = np.log(vix / vix.shift(1)).dropna()   # log-rendements (plus stables)

# Split train/test
vix_train = vix.loc[vix.index < pd.Timestamp(TEST_DATE)]
vix_test  = vix.loc[vix.index >= pd.Timestamp(TEST_DATE)]
ret_train = vix_ret.loc[vix_ret.index < pd.Timestamp(TEST_DATE)]
ret_test  = vix_ret.loc[vix_ret.index >= pd.Timestamp(TEST_DATE)]

# Variance réalisée à différentes fenêtres (pour HAR-RV et GARCH)
rv1d  = vix_ret.pow(2)                                 # proxy RV journalier
rv5d  = rv1d.rolling(5,  min_periods=3).mean()         # RV hebdo
rv22d = rv1d.rolling(22, min_periods=10).mean()        # RV mensuel

print(f"Train: {vix_train.index.min().date()} → {vix_train.index.max().date()} ({len(vix_train)} jours)")
print(f"Test : {vix_test.index.min().date()}  → {vix_test.index.max().date()}  ({len(vix_test)} jours)")
print(f"VIX train stats: mean={vix_train.mean():.2f}, std={vix_train.std():.2f}, max={vix_train.max():.2f}")

results = {}   # stocke les prédictions de tous les modèles


Train: 2000-08-11 → 2021-12-31 (5565 jours)
Test : 2022-01-03  → 2026-07-03  (1172 jours)
VIX train stats: mean=19.82, std=8.83, max=82.69


In [68]:
# =============================================================================
# MODÈLE 1 : HAR-RV (Heterogeneous AutoRegressive - Realized Variance)
# =============================================================================
# Corsi (2009). Prédit la variance réalisée future via 3 composantes :
#   RV_daily   : signal journalier (bruit haute fréquence)
#   RV_weekly  : composante hebdo (traders court terme)
#   RV_monthly : composante mensuelle (investisseurs long terme)
#
# Formule : RV_{t+h} = β₀ + β₁·RV_t + β₂·RV_{t-5:t} + β₃·RV_{t-22:t} + ε
#
# Ici on prédit la RV du VIX à h=1 et h=5 jours.
# Pertinence : HAR-RV est le benchmark non-ML standard pour la volatilité ;
# si ton ML ne bat pas HAR-RV, c'est le modèle à garder.
# =============================================================================

print("="*60)
print("MODÈLE 1 : HAR-RV")
print("="*60)

har_results = {}

for h in HORIZONS:
    # Cible : RV future à h jours (moyenne glissante des rendements²)
    rv_future = rv1d.shift(-h).rolling(h, min_periods=1).mean()

    X_cols = {"RV_1d": rv1d, "RV_5d": rv5d, "RV_22d": rv22d}
    df_har = pd.DataFrame(X_cols)
    df_har["RV_target"] = rv_future
    df_har = df_har.dropna()

    train_mask = df_har.index < pd.Timestamp(TEST_DATE)
    X_train = sm.add_constant(df_har.loc[train_mask, ["RV_1d","RV_5d","RV_22d"]])
    y_train = df_har.loc[train_mask, "RV_target"]
    X_test  = sm.add_constant(df_har.loc[~train_mask, ["RV_1d","RV_5d","RV_22d"]])
    y_test  = df_har.loc[~train_mask, "RV_target"]

    model = sm.OLS(y_train, X_train).fit()
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    corr = float(np.corrcoef(y_test, y_pred)[0,1])

    # Direction implicite : signe de la variation de RV prédit
    dir_true = (y_test.values > y_test.shift(1).reindex(y_test.index).values).astype(int)
    dir_pred = (y_pred.values  > y_test.shift(1).reindex(y_test.index).values).astype(int)
    valid = ~np.isnan(dir_true.astype(float))
    acc_dir = accuracy_score(dir_true[valid], dir_pred[valid])
    f1_dir  = f1_score(dir_true[valid], dir_pred[valid], average="macro", zero_division=0)

    print(f"\n[HAR-RV h={h}j]  R²={r2:.4f}  MAE={mae:.6f}  Corr={corr:.4f}")
    print(f"  Direction implicite : Acc={acc_dir:.4f}  F1={f1_dir:.4f}")
    print(f"  Coefficients : const={model.params['const']:.4f}, "
          f"β_daily={model.params['RV_1d']:.4f}, "
          f"β_weekly={model.params['RV_5d']:.4f}, "
          f"β_monthly={model.params['RV_22d']:.4f}")

    har_results[h] = {"R2": r2, "MAE": mae, "Corr": corr,
                       "Acc_dir": acc_dir, "F1_dir": f1_dir,
                       "y_pred": y_pred, "y_test": y_test,
                       "coefs": model.params.to_dict()}

results["HAR_RV"] = har_results


MODÈLE 1 : HAR-RV

[HAR-RV h=1j]  R²=0.0620  MAE=0.005897  Corr=0.2506
  Direction implicite : Acc=0.6704  F1=0.6470
  Coefficients : const=0.0027, β_daily=0.0941, β_weekly=0.2309, β_monthly=0.1315

[HAR-RV h=5j]  R²=0.0616  MAE=0.004132  Corr=0.2502
  Direction implicite : Acc=0.5175  F1=0.4925
  Coefficients : const=0.0033, β_daily=0.0537, β_weekly=0.1593, β_monthly=0.1254


In [85]:
# =============================================================================
# MODÈLE 2 : EGARCH(1,1) — rolling 1-step ahead, direction par variation de variance
# =============================================================================
# Direction prédite par variation de variance conditionnelle (Δσ²), pas son niveau.
# h>1 : simulation Monte-Carlo (500 trajectoires), médiane des variances simulées.
# =============================================================================

print("="*60)
print("MODÈLE 2 : EGARCH(1,1) — dist=skewt")
print("="*60)

garch_results = {}

# Fit initial sur le train
am_base = arch_model(ret_train * 100, vol="EGARCH", p=1, q=1,
                     dist="skewt", rescale=False)
res_base = am_base.fit(disp="off", show_warning=False)
print(f"  AIC={res_base.aic:.1f}  BIC={res_base.bic:.1f}")

for h in HORIZONS:
    try:
        history = ret_train.copy() * 100
        pred_var, pred_dates = [], []

        for date in ret_test.index:
            am  = arch_model(history, vol="EGARCH", p=1, q=1,
                             dist="skewt", rescale=False)
            res = am.fit(disp="off", show_warning=False, starting_values=res_base.params.values)

            if h == 1:
                fc     = res.forecast(horizon=1)
                sigma2 = fc.variance.iloc[-1, 0] / 10000
            else:
                # Simulation Monte-Carlo : 500 trajectoires, prend la médiane à l'horizon h
                sim    = res.forecast(horizon=h, method="simulation", simulations=500)
                sigma2 = sim.variance.iloc[-1, h-1] / 10000  # colonne h-1 = h-ième pas

            pred_var.append(sigma2)
            pred_dates.append(date)
            history.loc[date] = ret_test.loc[date] * 100

        pred_var  = pd.Series(pred_var, index=pred_dates)
        rv_actual = rv1d.reindex(pred_var.index)
        mask      = pred_var.notna() & rv_actual.notna()
        pred_var, rv_actual = pred_var[mask], rv_actual[mask]

        r2   = r2_score(rv_actual, pred_var)
        corr = float(np.corrcoef(rv_actual, pred_var)[0, 1])

        # Direction : variation de variance conditionnelle (Δσ²), pas son niveau
        var_change = pred_var.diff()
        y_pred = (var_change > 0).astype(int)
        y_true = (vix_ret.shift(-h).reindex(pred_var.index) > 0).astype(int)
        valid  = y_true.notna() & y_pred.notna()
        y_true, y_pred = y_true[valid], y_pred[valid]

        acc_dir = accuracy_score(y_true, y_pred)
        f1_dir  = f1_score(y_true, y_pred, average="macro", zero_division=0)

        print(f"\n[EGARCH h={h}j]  R²={r2:.4f}  Corr={corr:.4f}  Acc={acc_dir:.4f}  F1={f1_dir:.4f}")
        garch_results[h] = {"R2": r2, "Corr": corr, "Acc_dir": acc_dir, "F1_dir": f1_dir}

    except Exception as e:
        print(f"[EGARCH h={h}j] Échec : {e}")
        garch_results[h] = {"R2": np.nan, "Corr": np.nan, "Acc_dir": np.nan, "F1_dir": np.nan}

results["EGARCH"] = garch_results

MODÈLE 2 : EGARCH(1,1) — dist=skewt
  AIC=35931.2  BIC=35971.0

[EGARCH h=1j]  R²=0.0544  Corr=0.2344  Acc=0.5068  F1=0.4908

[EGARCH h=5j]  R²=0.0408  Corr=0.2064  Acc=0.5077  F1=0.4940


In [86]:
# =============================================================================
# MODÈLE 3 : Hidden Markov Model (HMM) — régimes appris, pas imposés
# =============================================================================
# Contrairement à tes seuils q33/q67 sur le niveau de VIX, le HMM apprend
# les régimes de façon non supervisée à partir des observations.
# States = régimes cachés (CALM/STRESS), observations = [vix_ret, vix_vol].
#
# Usage dual ici :
#   A) Classification des régimes sur le test (vs CALM/NORMAL/STRESS actuel)
#   B) Probabilité de transition → feature pour enrichir ton pipeline ML
#
# GaussianHMM(n_components=2) : 2 états, chacun avec sa propre gaussienne.
# Algorithme Baum-Welch (EM) pour l'estimation des paramètres.
# =============================================================================

print("="*60)
print("MODÈLE 3 : HMM — régimes appris")
print("="*60)

# Features observables : [log-rendement VIX, vol réalisée 5j, niveau VIX normalisé]
vix_norm = (vix - vix_train.mean()) / vix_train.std()

X_full = pd.DataFrame({
    "ret":    vix_ret,
    "vol5d":  np.sqrt(rv5d),          # vol réalisée (pas variance)
    "level":  vix_norm,
}).dropna()

X_tr = X_full.loc[X_full.index < pd.Timestamp(TEST_DATE)].values
X_te = X_full.loc[X_full.index >= pd.Timestamp(TEST_DATE)].values
idx_te = X_full.loc[X_full.index >= pd.Timestamp(TEST_DATE)].index

for n_states in [2, 3]:
    model_hmm = hmm.GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=200,
        random_state=RANDOM_STATE
    )
    model_hmm.fit(X_tr)

    # États cachés sur le test
    states_test = model_hmm.predict(X_te)
    # Probabilités de chaque état (soft assignment)
    proba_test  = model_hmm.predict_proba(X_te)   # shape (n_test, n_states)

    # Identifier quel état correspond à STRESS (celui avec la plus haute vol moyenne)
    state_vol = []
    for s in range(n_states):
        mask = (states_test == s)
        state_vol.append(np.sqrt(rv5d.reindex(idx_te).values[mask]).mean() if mask.sum() > 0 else 0)
    stress_state = int(np.argmax(state_vol))
    calm_state   = int(np.argmin(state_vol))

    # Régimes actuels (seuils VIX) pour comparaison
    q33 = vix_train.quantile(0.33)
    q67 = vix_train.quantile(0.67)
    vix_te_vals = vix.reindex(idx_te)
    regime_actual = np.where(vix_te_vals < q33, 0,
                    np.where(vix_te_vals >= q67, 2, 1))  # 0=CALM,1=NORM,2=STRESS

    hmm_stress_binary = (states_test == stress_state).astype(int)
    actual_stress_binary = (regime_actual == 2).astype(int)
    overlap = (hmm_stress_binary == actual_stress_binary).mean()

    print(f"\n[HMM K={n_states}]")
    print(f"  État STRESS identifié : état {stress_state} "
          f"(vol moy={state_vol[stress_state]:.4f})")
    print(f"  Overlap HMM-STRESS vs seuil-STRESS : {overlap:.3f}")
    print(f"  Distribution états test : {pd.Series(states_test).value_counts().to_dict()}")
    print(f"  Matrice de transition apprise :")
    trans = pd.DataFrame(model_hmm.transmat_,
                          index=[f"De état {i}" for i in range(n_states)],
                          columns=[f"Vers état {j}" for j in range(n_states)])
    print(trans.round(4).to_string())

    # Probabilité de stress comme feature — c'est ce qu'on injecterait dans XGBoost
    df_proba = pd.DataFrame(proba_test, index=idx_te,
                             columns=[f"P_state_{i}" for i in range(n_states)])
    df_proba[f"P_stress_K{n_states}"] = proba_test[:, stress_state]

    if n_states == 2:
        results["HMM"] = {"model": model_hmm, "proba_df": df_proba,
                           "stress_state": stress_state, "overlap": overlap}
        df_proba.to_csv(OUTPUT_DIR / "hmm_regime_proba.csv")
        print(f"  [SAVE] hmm_regime_proba.csv — à utiliser comme feature dans ton pipeline ML")

print("\n[HMM] La colonne P_stress_K2 est une feature continue [0,1] de stress,")
print("       plus riche que tes seuils VIX fixes. Injecte-la dans XGBoost/LightGBM.")


MODÈLE 3 : HMM — régimes appris

[HMM K=2]
  État STRESS identifié : état 1 (vol moy=0.0907)
  Overlap HMM-STRESS vs seuil-STRESS : 0.851
  Distribution états test : {0: 758, 1: 414}
  Matrice de transition apprise :
           Vers état 0  Vers état 1
De état 0       0.9710       0.0290
De état 1       0.0472       0.9528
  [SAVE] hmm_regime_proba.csv — à utiliser comme feature dans ton pipeline ML



[HMM K=3]
  État STRESS identifié : état 2 (vol moy=0.1159)
  Overlap HMM-STRESS vs seuil-STRESS : 0.759
  Distribution états test : {1: 499, 0: 436, 2: 237}
  Matrice de transition apprise :
           Vers état 0  Vers état 1  Vers état 2
De état 0       0.9752       0.0031       0.0217
De état 1       0.0079       0.9674       0.0247
De état 2       0.0348       0.0474       0.9177

[HMM] La colonne P_stress_K2 est une feature continue [0,1] de stress,
       plus riche que tes seuils VIX fixes. Injecte-la dans XGBoost/LightGBM.


In [87]:
# =============================================================================
# MODÈLE 4 : Filtre de Kalman — extraction du VIX "fondamental"
# =============================================================================
# Le VIX spot est vu comme une observation bruitée d'un processus latent
# (la "vraie" aversion au risque sous-jacente, non observable directement).
# Le filtre de Kalman sépare le signal du bruit.
#
# Modèle état-espace :
#   État caché    : x_t = x_{t-1} + w_t     (w ~ N(0, Q)) — marche aléatoire
#   Observation   : y_t = x_t + v_t          (v ~ N(0, R)) — bruit de mesure
#
# Sortie clé : x̂_t (état filtré = VIX lissé)
# Features dérivées :
#   vix_residual = VIX_spot - VIX_Kalman  → sur-extension par rapport au fondamental
#   vix_innovation = y_t - ŷ_{t|t-1}     → surprise par rapport à la prévision
# =============================================================================

print("="*60)
print("MODÈLE 4 : Filtre de Kalman")
print("="*60)

vix_array = vix.values.reshape(-1, 1)

# Estimation des paramètres par EM (Expectation-Maximization)
kf = KalmanFilter(
    transition_matrices=np.array([[1]]),         # x_t = x_{t-1} (RW)
    observation_matrices=np.array([[1]]),         # y_t = x_t + bruit
    initial_state_mean=np.array([vix.iloc[0]]),
    initial_state_covariance=np.array([[1.0]]),
    em_vars=["transition_covariance","observation_covariance"]
)

print("  Estimation EM des paramètres Q et R...")
kf = kf.em(vix_array, n_iter=20)
print(f"  Q (bruit d'état)       = {kf.transition_covariance[0,0]:.4f}")
print(f"  R (bruit d'observation)= {kf.observation_covariance[0,0]:.4f}")
print(f"  Ratio signal/bruit     = {kf.transition_covariance[0,0] / kf.observation_covariance[0,0]:.4f}")

# Filtrage : estimation de l'état caché à chaque pas
state_means, state_covs = kf.filter(vix_array)
vix_kalman = pd.Series(state_means[:, 0], index=vix.index, name="VIX_Kalman")

# Prédiction 1-step ahead (smoothing)
state_means_smooth, _ = kf.smooth(vix_array)
vix_smooth = pd.Series(state_means_smooth[:, 0], index=vix.index, name="VIX_Smooth")

# Features dérivées
vix_residual   = vix - vix_kalman           # écart spot vs fondamental
vix_innovation = vix - vix_smooth.shift(1)  # surprise vs prédiction précédente

# Évaluation : prédiction 1-step ahead vs valeur réelle sur le test
pred_test  = vix_smooth.shift(1).reindex(vix_test.index).dropna()
actual_test = vix.reindex(pred_test.index)

r2   = r2_score(actual_test, pred_test)
mae  = mean_absolute_error(actual_test, pred_test)
corr = float(np.corrcoef(actual_test, pred_test)[0,1])

# Direction : le VIX va-t-il monter ou descendre selon le filtre ?
dir_pred_kalman = (pred_test.diff() > 0).astype(int).dropna()
dir_actual      = (actual_test.diff() > 0).astype(int).reindex(dir_pred_kalman.index).dropna()
common = dir_pred_kalman.index.intersection(dir_actual.index)
acc_dir = accuracy_score(dir_actual.loc[common], dir_pred_kalman.loc[common])
f1_dir  = f1_score(dir_actual.loc[common], dir_pred_kalman.loc[common],
                   average="macro", zero_division=0)

print(f"\n[Kalman 1-step ahead]  R²={r2:.4f}  MAE={mae:.4f}  Corr={corr:.4f}")
print(f"  Direction : Acc={acc_dir:.4f}  F1={f1_dir:.4f}")
print(f"  VIX résiduel test : mean={vix_residual.reindex(vix_test.index).mean():.4f}, "
      f"std={vix_residual.reindex(vix_test.index).std():.4f}")

# Export des features Kalman (à injecter dans ton pipeline ML)
kalman_features = pd.DataFrame({
    "VIX_Kalman":     vix_kalman,
    "VIX_Residual":   vix_residual,      # spot - fondamental → mean-reversion signal
    "VIX_Innovation": vix_innovation,     # surprise → momentum signal
    "VIX_Smooth":     vix_smooth,
}).reindex(vix.index)

kalman_features.to_csv(OUTPUT_DIR / "kalman_vix_features.csv")
print("  [SAVE] kalman_vix_features.csv — VIX_Residual et VIX_Innovation à injecter en Phase 1 SHAP")

results["Kalman"] = {"R2": r2, "MAE": mae, "Corr": corr,
                      "Acc_dir": acc_dir, "F1_dir": f1_dir,
                      "features": kalman_features}


MODÈLE 4 : Filtre de Kalman
  Estimation EM des paramètres Q et R...
  Q (bruit d'état)       = 2.0641
  R (bruit d'observation)= 0.6108
  Ratio signal/bruit     = 3.3793

[Kalman 1-step ahead]  R²=0.9275  MAE=0.8833  Corr=0.9632
  Direction : Acc=0.5520  F1=0.5443
  VIX résiduel test : mean=-0.0002, std=0.3351
  [SAVE] kalman_vix_features.csv — VIX_Residual et VIX_Innovation à injecter en Phase 1 SHAP


In [88]:
# =============================================================================
# SYNTHÈSE COMPARATIVE
# =============================================================================

print("="*70)
print("COMPARAISON DES 4 MODÈLES (test 2022→2026)")
print("="*70)
print(f"{'Modèle':<22} {'Horizon':<8} {'R²':>8} {'Corr':>8} {'Acc_dir':>9} {'F1_dir':>8}")
print("─"*65)

summary_rows = []

for h in HORIZONS:
    har = results["HAR_RV"].get(h, {})
    egarch = results["EGARCH"].get(h, {})
    print(f"{'HAR-RV':<22} {h}j      {har.get('R2',float('nan')):>8.4f} "
          f"{har.get('Corr',float('nan')):>8.4f} {har.get('Acc_dir',float('nan')):>9.4f} "
          f"{har.get('F1_dir',float('nan')):>8.4f}")
    print(f"{'EGARCH(1,1)-skewt':<22} {h}j      {egarch.get('R2',float('nan')):>8.4f} "
          f"{egarch.get('Corr',float('nan')):>8.4f} {egarch.get('Acc_dir',float('nan')):>9.4f} "
          f"{egarch.get('F1_dir',float('nan')):>8.4f}")
    summary_rows.extend([
        {"Modele":"HAR-RV","Horizon":h,**har},
        {"Modele":"EGARCH","Horizon":h,**egarch}
    ])

kal = results["Kalman"]
print(f"{'Kalman (1-step)':<22} {'1j':<8} {kal.get('R2',float('nan')):>8.4f} "
      f"{kal.get('Corr',float('nan')):>8.4f} {kal.get('Acc_dir',float('nan')):>9.4f} "
      f"{kal.get('F1_dir',float('nan')):>8.4f}")
summary_rows.append({"Modele":"Kalman","Horizon":1,**kal})

hmm_r = results.get("HMM",{})
print(f"\n{'HMM (K=2)':<22} Overlap régimes vs seuils : {hmm_r.get('overlap',float('nan')):.4f}")
print("  P_stress → feature continue [0,1], voir hmm_regime_proba.csv")
print("  (HMM n'est pas un modèle de prédiction directe — c'est un enrichisseur de features)")

print("\n" + "="*70)
print("RECOMMANDATIONS D'INTÉGRATION DANS TON PIPELINE ML")
print("="*70)
print("""
1. HAR-RV    → benchmark baseline. Si ton ML (XGBoost/LightGBM) ne bat pas
               HAR-RV en R², c'est HAR-RV qu'on garde pour la composante RV.

2. EGARCH    → ajoute sigma²_t (variance conditionnelle) comme feature dans
               ton Phase 1 SHAP. Signal prospectif : la variance conditionnelle
               EGARCH à t est calculable AVANT d'observer t+1.

3. HMM       → remplace tes seuils q33/q67 par P_stress (probabilité continue
               d'être en état stress selon le HMM). Plus riche, capture les
               transitions progressives que le seuil dur rate.

4. Kalman    → ajoute VIX_Residual et VIX_Innovation dans features_post_engineering.
               VIX_Residual > 0 : VIX sur-étendu → mean-reversion prédictible.
               VIX_Innovation : surprise du jour → momentum à court terme.

Prochaine étape suggérée : re-run Phase 1 SHAP avec ces 4 nouvelles features
(EGARCH_cond_var, HMM_P_stress, Kalman_residual, Kalman_innovation).
""")

# Export Excel
df_summary = pd.DataFrame([r for r in summary_rows
                            if all(k in r for k in ["R2","Corr","Acc_dir","F1_dir"])])
df_summary.to_excel(OUTPUT_DIR / "timeseries_models_comparison.xlsx", index=False)
print("[SAVE] timeseries_models_comparison.xlsx")
print("[NOTE] Aucun modèle enregistré — validation explicite requise.")


COMPARAISON DES 4 MODÈLES (test 2022→2026)
Modèle                 Horizon        R²     Corr   Acc_dir   F1_dir
─────────────────────────────────────────────────────────────────
HAR-RV                 1j        0.0620   0.2506    0.6704   0.6470
EGARCH(1,1)-skewt      1j        0.0544   0.2344    0.5068   0.4908
HAR-RV                 5j        0.0616   0.2502    0.5175   0.4925
EGARCH(1,1)-skewt      5j        0.0408   0.2064    0.5077   0.4940
Kalman (1-step)        1j         0.9275   0.9632    0.5520   0.5443

HMM (K=2)              Overlap régimes vs seuils : 0.8507
  P_stress → feature continue [0,1], voir hmm_regime_proba.csv
  (HMM n'est pas un modèle de prédiction directe — c'est un enrichisseur de features)

RECOMMANDATIONS D'INTÉGRATION DANS TON PIPELINE ML

1. HAR-RV    → benchmark baseline. Si ton ML (XGBoost/LightGBM) ne bat pas
               HAR-RV en R², c'est HAR-RV qu'on garde pour la composante RV.

2. EGARCH    → ajoute sigma²_t (variance conditionnelle) comme feat